# Model Training and Validation
## Project: House Price Prediction using Linear Regression

This notebook covers the training, validation, and serialization of our house price prediction model. We:
1. Train-test split the cleaned data (80/20).
2. Train a Linear Regression estimator combined with the Preprocessor in a single Scikit-learn `Pipeline` object.
3. Compute core evaluation metrics: MAE, MSE, RMSE, and R² Score.
4. Inspect model parameters: Intercept and Feature Weights.
5. Plot regression diagnostics: Actual vs Predicted, Residuals, and Error Distribution.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

# Ensure project root in path
sys.path.append(str(Path(os.getcwd()).resolve().parent))
import config
from src.data_loader import DataLoader
from src.feature_engineering import clean_data
from src.preprocessing import build_preprocessing_pipeline

sns.set_theme(style="whitegrid")

### 1. Load and Split Data
We load, clean, and split the dataset.

In [ ]:
loader = DataLoader()
df_raw = loader.load_data(config.TRAIN_PATH, is_training=True)
df_cleaned = clean_data(df_raw, is_training=True)
X, y = loader.split_features_target(df_cleaned)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]}, Validation size: {X_val.shape[0]}")

### 2. Fit the Full Pipeline
We combine the preprocessing step and the Linear Regression estimator into a unified Pipeline to prevent training-serving leakages.

In [ ]:
preprocessor = build_preprocessing_pipeline()
regressor = LinearRegression(**config.MODEL_PARAMS)

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", regressor)
])

model_pipeline.fit(X_train, y_train)
print("Model pipeline fitted successfully!")

### 3. Model Interpretation: Feature Weights
We extract the fitted parameters to identify the impact of each feature on house prices.

In [ ]:
fitted_preprocessor = model_pipeline.named_steps["preprocessor"]
fitted_regressor = model_pipeline.named_steps["regressor"]

feature_names = [name.split("__")[-1] for name in fitted_preprocessor.get_feature_names_out()]
coefficients = fitted_regressor.coef_
intercept = fitted_regressor.intercept_

weights_df = pd.DataFrame({
    "Feature": feature_names,
    "Weight (Coefficient)": coefficients
}).sort_values(by="Weight (Coefficient)", ascending=False)

print(f"Model Intercept (Base Price): ${intercept:,.2f}")
print("\nFeature weights on standardized features:")
weights_df

### 4. Run Model Predictions & Evaluations
We run inference on our validation set and calculate evaluation metrics.

In [ ]:
from src.evaluate import calculate_metrics

y_pred = model_pipeline.predict(X_val)
metrics = calculate_metrics(y_val, y_pred)

### 5. Regression Diagnostic Plots
We render the performance graphs: Actual vs. Predicted, Residual distribution, and Error margins.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 6))

# 1. Actual vs Predicted
sns.scatterplot(x=y_val, y=y_pred, alpha=0.7, color="#4F46E5", ax=ax[0])
min_val = min(y_val.min(), y_pred.min())
max_val = max(y_val.max(), y_pred.max())
ax[0].plot([min_val, max_val], [min_val, max_val], color="#EF4444", linestyle="--", label="Perfect Fit")
ax[0].set_title("Actual vs. Predicted House Prices", fontweight="bold")
ax[0].set_xlabel("Actual Price ($)")
ax[0].set_ylabel("Predicted Price ($)")
ax[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))
ax[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))
ax[0].legend()

# 2. Residual Plot
residuals = y_val - y_pred
sns.scatterplot(x=y_pred, y=residuals, alpha=0.7, color="#10B981", ax=ax[1])
ax[1].axhline(y=0, color="#EF4444", linestyle="--")
ax[1].set_title("Residuals vs. Predicted Price", fontweight="bold")
ax[1].set_xlabel("Predicted Price ($)")
ax[1].set_ylabel("Residuals / Errors ($)")
ax[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))
ax[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))

# 3. Error Distribution
sns.histplot(residuals, kde=True, color="#6366F1", ax=ax[2], bins=20)
ax[2].axvline(x=0, color="#EF4444", linestyle="--")
ax[2].set_title("Distribution of Prediction Errors (Residuals)", fontweight="bold")
ax[2].set_xlabel("Prediction Error ($)")
ax[2].set_ylabel("Count")
ax[2].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))

plt.tight_layout()
plt.show()

### 6. Pipeline Serialization
We serialize the fitted `model_pipeline` using `joblib` so that it can be loaded for real-time predictions.

In [ ]:
import joblib

# Save pipeline
export_path = config.MODEL_DIR / "model_notebook_test.joblib"
joblib.dump(model_pipeline, export_path)
print(f"Notebook test model saved successfully at: {export_path}")